---
execute:
  skip: true
authors:
- name: Fernanda Ribeiro
doi: 10.5281/zenodo.18740132
github: https://github.com/neurodesk/neurodeskedu/blob/main/books/examples/functional_imaging/deepretinotopy.ipynb
edit_url: https://github.com/neurodesk/neurodeskedu/edit/main/books/examples/functional_imaging/deepretinotopy.ipynb
downloads:
- url: /edu/_sources/examples/functional_imaging/deepretinotopy.ipynb
  title: Download source notebook
  filename: deepretinotopy.ipynb
  static: false
---
# DeepRetinotopy Toolbox

:::{admonition} Reviewed
:class: nd-review-badge nd-review-badge--reviewed

[review issue](<https://github.com/neurodesk/neurodeskedu-reviews/issues/10>) · [@MoniDoerig](<https://github.com/MoniDoerig>)
:::

:::{dropdown} Run this notebook
:class: nd-launch

- [Run in Australia](<https://play.neurodesk.cloud.edu.au/hub/user-redirect/git-pull?repo=https%3A//github.com/neurodesk/neurodeskedu&urlpath=lab/tree/neurodeskedu/books/examples/functional_imaging/deepretinotopy.ipynb&branch=main>)
- [Run in Europe](<https://play-europe.neurodesk.org/hub/user-redirect/git-pull?repo=https%3A//github.com/neurodesk/neurodeskedu&urlpath=lab/tree/neurodeskedu/books/examples/functional_imaging/deepretinotopy.ipynb&branch=main>)
- [Run in America](<https://play-america.neurodesk.org/hub/user-redirect/git-pull?repo=https%3A//github.com/neurodesk/neurodeskedu&urlpath=lab/tree/neurodeskedu/books/examples/functional_imaging/deepretinotopy.ipynb&branch=main>)
- [Run in Global Server 1](<https://play-europe.neurodesk.org/hub/user-redirect/git-pull?repo=https%3A//github.com/neurodesk/neurodeskedu&urlpath=lab/tree/neurodeskedu/books/examples/functional_imaging/deepretinotopy.ipynb&branch=main>)
- [Run in Global Server 2](<https://edu.neurodesk.org/hub/user-redirect/git-pull?repo=https%3A//github.com/neurodesk/neurodeskedu&urlpath=lab/tree/neurodeskedu/books/examples/functional_imaging/deepretinotopy.ipynb&branch=main>)
:::



**Author**: Fernanda Ribeiro

**Date**: 12 Nov 2025

**License:** 
<div style="margin-top: 10px;">
    <a href="https://opensource.org/licenses/MIT" target="_blank" style="color: #0066cc;">
        <i class="fas fa-balance-scale"></i> MIT License
    </a>
</div>


**Note:** If this notebook uses neuroimaging tools from Neurocontainers, those tools retain their original licenses. Please see <a href="https://neurodesk.org/overview/how-to-cite-us/" target="_blank" style="color: #0066cc;">Neurodesk citation guidelines</a> for details.

### Citation and Resources
#### Tools included in this workflow
__deepretinotopy__:
- Ribeiro, F. L. et al. (2025). Predicting functional topography of the human visual cortex from cortical anatomy at scale. [bioRxiv](https://www.biorxiv.org/content/10.1101/2025.11.27.690210v2)

#### Dataset
__NYU Retinotopy Dataset__

Himmelberg, Marc M. and Kurzawski, Jan W. and Benson, Noah C. and Carrasco, Marisa and Winawer, Jonathan. “NYU Retinotopy dataset” 2021. [doi:10.18112/openneuro.ds003787.v1.0.1](https://github.com/OpenNeuroDatasets/ds003787.git)

## DeepRetinotopy toolbox

This interactive notebook demonstrates how to predict retinotopic maps from anatomical data using a geometric deep learning model of retinotopy. For more details about the toolbox, please check our [repository](https://github.com/felenitaribeiro/deepRetinotopy_Toolbox). The accompanying manuscript will be available soon.

Requirements:

1. freesurfer directory;
2. HCP "fs_LR-deformed_to-fsaverage" surfaces, available [here](https://github.com/Washington-University/HCPpipelines/tree/master/global/templates/standard_mesh_atlases/resample_fsaverage).


## Load Software Tools

In [1]:
%%capture 
!pip install nilearn==0.13.1 nibabel==5.3.3 numpy==2.3.5

In [2]:
# we can use module to load deepretinotopy in a specific version
import module
import os
await module.load('deepretinotopy/1.0.18')
await module.list()

['deepretinotopy/1.0.18']

In [3]:
# Request a freesurfer license and store it in your homedirectory. This is just an example - please replace with your license id:
!echo "Steffen.Bollmann@cai.uq.edu.au" > ~/.license
!echo "21029" >> ~/.license
!echo "*Cqyn12sqTCxo" >> ~/.license
!echo "FSxgcvGkNR59Y" >> ~/.license

## Download data

Let's start by downloading some data available on OpenNeuro. DeepRetinotopy only requires a minimal subset of FreeSurfer outputs, which means you can significantly reduce storage requirements by downloading only the following files per subject:

### Essential Files
```
<subject_id>/surf/
├── lh.white          # White matter surface
├── lh.pial           # Pial surface  
├── lh.sphere.reg     # Spherical surface for resampling
├── rh.white          # White matter surface
├── rh.pial           # Pial surface
└── rh.sphere.reg     # Spherical surface for resampling
```
**_Note_**: The toolbox automatically generates the midthickness surfaces and curvature data from these base files during processing. All other FreeSurfer outputs (thickness maps, parcellations, etc.) are not required for retinotopic mapping prediction.

In [4]:
!mkdir -p ./data

os.chdir('./data')
!datalad install https://github.com/OpenNeuroDatasets/ds003787.git # install NYU retinotopy dataset
os.chdir('ds003787/derivatives/freesurfer') # change directories
list_of_subs = os.listdir('.')

# download required files to run deepretinotopy
for sub in list_of_subs:
    for hemisphere in ['lh', 'rh']:
        for file in ['pial', 'white', 'sphere', 'sphere.reg', 'thickness']:
            cmd=f"""datalad get {sub}/surf/{hemisphere}.{file}""" 
            os.system(cmd)

Cloning:   0%|                             | 0.00/2.00 [00:00<?, ? candidates/s]


Enumerating: 0.00 Objects [00:00, ? Objects/s]
                                              
Counting:   0%|                               | 0.00/137k [00:00<?, ? Objects/s]


                                                                                
Compressing:   0%|                            | 0.00/113k [00:00<?, ? Objects/s]


                                                                                
Receiving:   0%|                              | 0.00/137k [00:00<?, ? Objects/s]


Receiving:  18%|███▉                  | 24.6k/137k [00:00<00:00, 242k Objects/s]

...

get(ok): derivatives/freesurfer/sub-wlsubj126/surf/rh.pial (file) [from s3-PUBLIC...]


get(ok): derivatives/freesurfer/sub-wlsubj126/surf/rh.white (file) [from s3-PUBLIC...]


get(ok): derivatives/freesurfer/sub-wlsubj126/surf/rh.sphere (file) [from s3-PUBLIC...]


get(ok): derivatives/freesurfer/sub-wlsubj126/surf/rh.sphere.reg (file) [from s3-PUBLIC...]


get(ok): derivatives/freesurfer/sub-wlsubj126/surf/rh.thickness (file) [from s3-PUBLIC...]


Since we have preprocessed data available, the only remaining requirements are the HCP template surfaces. 
Thus, we need to create a "templates" folder (for example, ./templates/) and download the following files from the link above:

- "fs_LR-deformed_to-fsaverage.R.sphere.32k_fs_LR.surf.gii";
- "fs_LR-deformed_to-fsaverage.L.sphere.32k_fs_LR.surf.gii";

In [5]:
# Set up paths relative to current directory
os.environ['PATH_FREESURFER_DIR'] = "."
os.environ['PATH_HCP_TEMPLATE_SURFACE'] = "../../templates"
os.environ['DATASET_NAME'] = "nyu"
os.environ['MAPS'] = "polarAngle,eccentricity"

In [6]:
# Create templates directory
os.makedirs(os.environ['PATH_HCP_TEMPLATE_SURFACE'], exist_ok=True)

In [7]:
# Download templates to the relative path
!cd $PATH_HCP_TEMPLATE_SURFACE && \
wget -nc https://github.com/Washington-University/HCPpipelines/raw/master/global/templates/standard_mesh_atlases/resample_fsaverage/fs_LR-deformed_to-fsaverage.L.sphere.32k_fs_LR.surf.gii && \
wget -nc https://github.com/Washington-University/HCPpipelines/raw/master/global/templates/standard_mesh_atlases/resample_fsaverage/fs_LR-deformed_to-fsaverage.R.sphere.32k_fs_LR.surf.gii

--2026-07-08 07:58:07--  https://github.com/Washington-University/HCPpipelines/raw/master/global/templates/standard_mesh_atlases/resample_fsaverage/fs_LR-deformed_to-fsaverage.L.sphere.32k_fs_LR.surf.gii
Resolving github.com (github.com)... 4.237.22.38
Connecting to github.com (github.com)|4.237.22.38|:443... 

connected.
HTTP request sent, awaiting response... 

302 Found
Location: https://raw.githubusercontent.com/Washington-University/HCPpipelines/master/global/templates/standard_mesh_atlases/resample_fsaverage/fs_LR-deformed_to-fsaverage.L.sphere.32k_fs_LR.surf.gii [following]
--2026-07-08 07:58:08--  https://raw.githubusercontent.com/Washington-University/HCPpipelines/master/global/templates/standard_mesh_atlases/resample_fsaverage/fs_LR-deformed_to-fsaverage.L.sphere.32k_fs_LR.surf.gii
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 

200 OK
Length: 720487 (704K) [text/plain]
Saving to: ‘fs_LR-deformed_to-fsaverage.L.sphere.32k_fs_LR.surf.gii’

fs_LR-deformed_to-f 100%[===================>] 703.60K  --.-KB/s    in 0.01s   

2026-07-08 07:58:08 (69.8 MB/s) - ‘fs_LR-deformed_to-fsaverage.L.sphere.32k_fs_LR.surf.gii’ saved [720487/720487]

--2026-07-08 07:58:08--  https://github.com/Washington-University/HCPpipelines/raw/master/global/templates/standard_mesh_atlases/resample_fsaverage/fs_LR-deformed_to-fsaverage.R.sphere.32k_fs_LR.surf.gii
Resolving github.com (github.com)... 

4.237.22.38
Connecting to github.com (github.com)|4.237.22.38|:443... connected.
HTTP request sent, awaiting response... 

302 Found
Location: https://raw.githubusercontent.com/Washington-University/HCPpipelines/master/global/templates/standard_mesh_atlases/resample_fsaverage/fs_LR-deformed_to-fsaverage.R.sphere.32k_fs_LR.surf.gii [following]
--2026-07-08 07:58:09--  https://raw.githubusercontent.com/Washington-University/HCPpipelines/master/global/templates/standard_mesh_atlases/resample_fsaverage/fs_LR-deformed_to-fsaverage.R.sphere.32k_fs_LR.surf.gii
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 

200 OK
Length: 707932 (691K) [text/plain]
Saving to: ‘fs_LR-deformed_to-fsaverage.R.sphere.32k_fs_LR.surf.gii’

fs_LR-deformed_to-f 100%[===================>] 691.34K  --.-KB/s    in 0.01s   

2026-07-08 07:58:09 (62.1 MB/s) - ‘fs_LR-deformed_to-fsaverage.R.sphere.32k_fs_LR.surf.gii’ saved [707932/707932]



## Running deepRetinotopy for a single subject

Here, we illustrate how to run deepRetinotopy for a single subject. Note that, if no output directory is provided, all maps are saved within each subject subdirectory in the freesurfer directory. 

In [8]:
## Process single subject without an explicit output directory
!deepRetinotopy -s $PATH_FREESURFER_DIR -t $PATH_HCP_TEMPLATE_SURFACE -d $DATASET_NAME -m $MAPS -i sub-wlsubj001

Processing single subject: sub-wlsubj001


Log files location: /home/jovyan/workspace/books/examples/functional_imaging/data/ds003787/derivatives/freesurfer/logs
Output log: deepRetinotopy_20260708_075811_output.log
Error log: deepRetinotopy_20260708_075811_error.log
--------------------------------------------------------------------------------
DeepRetinotopy: retinotopic mapping using deep learning
--------------------------------------------------------------------------------
Log files: /home/jovyan/workspace/books/examples/functional_imaging/data/ds003787/derivatives/freesurfer/logs/deepRetinotopy_20260708_075811_output.log and deepRetinotopy_20260708_075811_error.log
--------------------------------------------------------------------------------
Directories and parameters:

Path to FreeSurfer directory: /home/jovyan/workspace/books/examples/functional_imaging/data/ds003787/derivatives/freesurfer
Path to fs_LR-deformed_to-fsaverage template surfaces: /home/jovyan/workspace/books/examples/functional_imaging/data/ds003787/

Saving /home/jovyan/workspace/books/examples/functional_imaging/data/ds003787/derivatives/freesurfer/sub-wlsubj001/surf/lh.white.gii as a surface


Saving /home/jovyan/workspace/books/examples/functional_imaging/data/ds003787/derivatives/freesurfer/sub-wlsubj001/surf/lh.pial.gii as a surface
[sub-wlsubj001] Generating midthickness surface...


Midthickness surface saved at /home/jovyan/workspace/books/examples/functional_imaging/data/ds003787/derivatives/freesurfer/sub-wlsubj001/surf/lh.graymid.gii


...

[Step 3] Resampling predictions to native space...


Model: model
Map: eccentricity
Hemisphere: rh
Processing single subject: sub-wlsubj001
Output mode: In-place (within FreeSurfer directory structure)
Processing subject: sub-wlsubj001
=== Processing Step 3 for subject: sub-wlsubj001 ===
[sub-wlsubj001] Using FreeSurfer directory: /home/jovyan/workspace/books/examples/functional_imaging/data/ds003787/derivatives/freesurfer/sub-wlsubj001/deepRetinotopy
[sub-wlsubj001] Resampling ROI from fsaverage to native space for the right hemisphere...


[sub-wlsubj001] Resampling predicted map from fsaverage to native space for the right hemisphere...


[sub-wlsubj001] Applying mask to the predicted map of the right hemisphere...




=== Subject sub-wlsubj001 completed in 0 minutes ===

[Step 3] COMPLETED!
Total execution time: 0m 3s
Subject processed: sub-wlsubj001
Map: eccentricity | Model: model | Hemisphere: rh
Output location: In-place within FreeSurfer directory
eccentricity resampling completed for rh hemisphere
--------------------------------------------------------------------------------
Execution completed!


In [9]:
!ls $PATH_FREESURFER_DIR/sub-wlsubj001/deepRetinotopy

sub-wlsubj001.ROI.lh.native.label.gii
sub-wlsubj001.ROI.rh.native.label.gii
sub-wlsubj001.fs_predicted_eccentricity_lh_curvatureFeat_model.func.gii
sub-wlsubj001.fs_predicted_eccentricity_rh_curvatureFeat_model.func.gii
sub-wlsubj001.fs_predicted_polarAngle_lh_curvatureFeat_model.func.gii
sub-wlsubj001.fs_predicted_polarAngle_rh_curvatureFeat_model.func.gii
sub-wlsubj001.predicted_eccentricity_model.lh.native.func.gii
sub-wlsubj001.predicted_eccentricity_model.rh.native.func.gii
sub-wlsubj001.predicted_polarAngle_model.lh.native.func.gii
sub-wlsubj001.predicted_polarAngle_model.rh.native.func.gii


In [10]:
## Process single subject with an explicit output directory
! mkdir -p ../../../output && \
deepRetinotopy -s $PATH_FREESURFER_DIR -t $PATH_HCP_TEMPLATE_SURFACE -d $DATASET_NAME -m $MAPS -i sub-wlsubj001 -o ../../../output

Processing single subject: sub-wlsubj001
Log files location: /home/jovyan/workspace/books/examples/functional_imaging/data/output
Output log: deepRetinotopy_20260708_080424_output.log
Error log: deepRetinotopy_20260708_080424_error.log
--------------------------------------------------------------------------------
DeepRetinotopy: retinotopic mapping using deep learning
--------------------------------------------------------------------------------
Log files: /home/jovyan/workspace/books/examples/functional_imaging/data/output/deepRetinotopy_20260708_080424_output.log and deepRetinotopy_20260708_080424_error.log
--------------------------------------------------------------------------------
Directories and parameters:

Path to FreeSurfer directory: /home/jovyan/workspace/books/examples/functional_imaging/data/ds003787/derivatives/freesurfer
Path to fs_LR-deformed_to-fsaverage template surfaces: /home/jovyan/workspace/books/examples/functional_imaging/data/ds003787/templates
Dataset n

Hemisphere: lh
Processing single subject: sub-wlsubj001
Output directory: /home/jovyan/workspace/books/examples/functional_imaging/data/output
Processing subject: sub-wlsubj001
=== Processing Step 1 for subject: sub-wlsubj001 ===
[sub-wlsubj001] Using custom output directory: /home/jovyan/workspace/books/examples/functional_imaging/data/output/sub-wlsubj001
[sub-wlsubj001] Fast mode enabled.
[sub-wlsubj001] [Step 1.1] Generating mid-thickness surface and curvature data if not available...
[sub-wlsubj001] Converting surfaces...


Saving /home/jovyan/workspace/books/examples/functional_imaging/data/output/sub-wlsubj001/surf/lh.white.gii as a surface


Saving /home/jovyan/workspace/books/examples/functional_imaging/data/output/sub-wlsubj001/surf/lh.pial.gii as a surface
[sub-wlsubj001] Generating midthickness surface...


Midthickness surface saved at /home/jovyan/workspace/books/examples/functional_imaging/data/output/sub-wlsubj001/surf/lh.graymid.gii


...

[Step 3] Resampling predictions to native space...


Model: model
Map: eccentricity
Hemisphere: rh
Processing single subject: sub-wlsubj001
Output directory: /home/jovyan/workspace/books/examples/functional_imaging/data/output
Processing subject: sub-wlsubj001
=== Processing Step 3 for subject: sub-wlsubj001 ===
[sub-wlsubj001] Using custom output directory: /home/jovyan/workspace/books/examples/functional_imaging/data/output/sub-wlsubj001
[sub-wlsubj001] Resampling ROI from fsaverage to native space for the right hemisphere...


[sub-wlsubj001] Resampling predicted map from fsaverage to native space for the right hemisphere...


[sub-wlsubj001] Applying mask to the predicted map of the right hemisphere...


=== Subject sub-wlsubj001 completed in 0 minutes ===

[Step 3] COMPLETED!
Total execution time: 0m 2s
Subject processed: sub-wlsubj001
Map: eccentricity | Model: model | Hemisphere: rh
Output location: /home/jovyan/workspace/books/examples/functional_imaging/data/output
eccentricity resampling completed for rh hemisphere
--------------------------------------------------------------------------------
Execution completed!
All output files have been saved to: /home/jovyan/workspace/books/examples/functional_imaging/data/output


In [11]:
!ls ../../../output/sub-wlsubj001/

deepRetinotopy	surf


## Running deepRetinotopy for multiple subjects

If no subject is specified, all subjects within the FreeSurfer directory will be processed. However, for optimal speed, parallelising the processing for multiple subjects across multiple cores is preferred.

In [12]:
## Determine how many cores you have available
os.environ['NUMBER_OF_CORES'] = str(os.cpu_count() - 1)

In [13]:
%%bash

SUBJECT_LIST=(sub-wlsubj001 sub-wlsubj004 sub-wlsubj006 sub-wlsubj007)

process_subject() {
    local subject=$1
    echo "Processing subject: $subject"
    deepRetinotopy -s $PATH_FREESURFER_DIR -t $PATH_HCP_TEMPLATE_SURFACE -d $DATASET_NAME -m $MAPS -i "$subject"
}

export -f process_subject

printf "%s\n" "${SUBJECT_LIST[@]}" | xargs -P "$NUMBER_OF_CORES" -I {} bash -c 'process_subject "{}"'

Processing subject: sub-wlsubj001


Processing subject: sub-wlsubj004
Processing subject: sub-wlsubj006


Processing subject: sub-wlsubj007


Processing single subject: sub-wlsubj007


Processing single subject: sub-wlsubj004


...


[Step 3] COMPLETED!
Total execution time: 0m 3s
Sub

ject processed: sub-wlsubj004
Map: eccentricity | Model: model | Hemisphere: rh
Output location: In-

place within FreeSurfer directory


eccentricity resampling completed for rh hemisphere
------------------------------------------------

--------------------------------
Execution completed!


In [14]:
!ls ./sub-wlsubj004/deepRetinotopy

sub-wlsubj004.ROI.lh.native.label.gii
sub-wlsubj004.ROI.rh.native.label.gii
sub-wlsubj004.fs_predicted_eccentricity_lh_curvatureFeat_model.func.gii
sub-wlsubj004.fs_predicted_eccentricity_rh_curvatureFeat_model.func.gii
sub-wlsubj004.fs_predicted_polarAngle_lh_curvatureFeat_model.func.gii
sub-wlsubj004.fs_predicted_polarAngle_rh_curvatureFeat_model.func.gii
sub-wlsubj004.predicted_eccentricity_model.lh.native.func.gii
sub-wlsubj004.predicted_eccentricity_model.rh.native.func.gii
sub-wlsubj004.predicted_polarAngle_model.lh.native.func.gii
sub-wlsubj004.predicted_polarAngle_model.rh.native.func.gii


## Maps visualization

Finally, we can visualize the predicted maps in the individual's native space using nilearn.

In [15]:
from nilearn import plotting
from ipywidgets import Dropdown, interact
import nibabel as nib
import numpy as np
from IPython.display import display


def plot_data(subject, retinotopic_map):
    background = np.array(nib.load(f"./{subject}/surf/lh.graymid.H.gii").agg_data())[:]
    background[background < 0] = -1
    background[background > 1] = 1
    threshold = 1
    data = np.array(nib.load(f"./{subject}/deepRetinotopy/{subject}.predicted_{retinotopic_map}_model.lh.native.func.gii").agg_data())[:]
    data[data != 0] = data[data != 0] + threshold
    if retinotopic_map == 'eccentricity':
        max_value = 8
    else: 
        max_value = 360
    view = plotting.view_surf(
        surf_mesh=f"./{subject}/surf/lh.midthickness.surf.gii",
        surf_map=np.reshape(data[:], (-1)),
        cmap="gist_rainbow_r", black_bg=False, symmetric_cmap=False, bg_map = background,
        threshold=threshold, vmax=max_value + threshold, vmin = threshold)
    return view


<div style="background-color: light-dark(#fff3cd, #4a3800); border-left: 4px solid #ff9800; padding: 15px; margin: 20px 0; color: light-dark(#856404, #ffd966);">
  <p style="margin:0; color: light-dark(#856404, #ffd966);">
    ⚠️ <strong>Note:</strong> 
    <code style="background-color: light-dark(#ffe69c, #332600); padding:2px 4px; border-radius:3px;">ipywidgets</code> 
    that rely on Python callbacks 
    (<code style="background-color: light-dark(#ffe69c, #332600); padding:2px 4px; border-radius:3px;">observe</code>, 
    <code style="background-color: light-dark(#ffe69c, #332600); padding:2px 4px; border-radius:3px;">on_click</code>) 
    require a running kernel and do not function in the static HTML version of this notebook. 
    Widgets using client-side trait linking 
    (<code style="background-color: light-dark(#ffe69c, #332600); padding:2px 4px; border-radius:3px;">jslink</code>) 
    remain interactive without a kernel.
  </p>
</div>

In [16]:
subjects = ['sub-wlsubj001', 'sub-wlsubj004', 'sub-wlsubj006', 'sub-wlsubj007']
retinotopic_maps = ['eccentricity', 'polarAngle']

@interact(subject = subjects, retinotopic_map = retinotopic_maps)
def plot1(subject, retinotopic_map):
    view = plot_data(subject, retinotopic_map)
    display(view)

#### Dependencies in Jupyter/Python
- Using the package [watermark](https://github.com/rasbt/watermark) to document system environment and software versions used in this notebook, alongside the Neurodesktop version extracted from the `JUPYTER_IMAGE` or `NEURODESKTOP_VERSION` environment variables.

In [17]:
import os

%load_ext watermark

%watermark
%watermark --iversions

neurodesktop_version = (
    os.environ.get('JUPYTER_IMAGE', '').split(':')[-1] or
    os.environ.get('NEURODESKTOP_VERSION', 'unknown')
)

print(f"Neurodesktop version: {neurodesktop_version}")

Last updated: 2026-07-08T08:11:23.644260+00:00

Python implementation: CPython
Python version       : 3.13.13
IPython version      : 9.12.0

Compiler    : GCC 14.3.0
OS          : Linux
Release     : 6.8.0-111-generic
Machine     : x86_64
Processor   : x86_64
CPU cores   : 16
Architecture: 64bit

IPython   : 9.12.0
ipywidgets: 8.1.8
nibabel   : 5.3.3
nilearn   : 0.13.1
numpy     : 2.3.5

Neurodesktop version: 2026-06-04
